# Figure: Melt Volatile Degassing Paths (P-normalized)

In [ ]:
from pathlib import Path

results_directory = Path().resolve().parent / "Model_Outputs"
SAVE_FIG = True

## Import data and styling

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from helpers.plot_styles import (
    PLOTLY_TICK_LEN,
    PLOTLY_FONT,
    PLOTLY_LEGEND_FONTSIZE,
    PLOTLY_TICK_FONTSIZE,
    SAMPLE_DISPLAY_NAMES,
    TOOL_COLORS_HEX,
    TOOL_LINE_STYLE,
)
from helpers.degassing_data import load_all_systems

# --- USER INPUTS --- #
SAMPLES = ["MORB", "Kilauea", "Fuego", "Fogo"]
TOOLS  = ["DCompress", "DCompress (IM)", "EVo", "MAGEC", "SulfurX", "VolFe", "VESIcal_Iacono"]

# Row definitions: (DataFrame column, y-axis label).
Y_ROWS = [
    ("H2OT_m_wtpc", "H<sub>2</sub>O<sup>melt</sup> total (wt%)"),
    ("CO2T_m_ppmw", "CO<sub>2</sub><sup>melt</sup> total (ppm)"),
    ("ST_m_ppmw",   "S<sup>melt</sup> total (ppm)"),
]


In [ ]:
systems = load_all_systems(SAMPLES, TOOLS, results_dir=results_directory)

## Build the figure
Here we set the x-axis limit to that of the onset of significant degassing for each subplot to best show how trends diverge.

In [ ]:
CHANGE_TOL = 0.05  # fraction of a curve's total change that counts as "still changing"
X_PAD      = 1.05  # show a touch of the flat region to the right of the crop point

def panel_x_max(sample, col_name, tol=CHANGE_TOL):
    """Largest P/P_i at which any tool in this panel still deviates from its initial
    (highest-P) value by more than `tol` of that curve's total change. To the right of
    this point every curve is essentially flat, so it is a sensible x-axis maximum."""
    candidates = []
    for tool in TOOLS:
        df = systems.get(sample, {}).get(tool)
        if df is None or col_name not in df.columns or "P_bars" not in df.columns:
            continue
        p_init = df["P_bars"].iloc[0]
        if p_init == 0:
            continue
        y = df[col_name]
        total = abs(y.iloc[-1] - y.iloc[0])
        if total == 0:
            continue
        dev = (y - y.iloc[0]).abs() / total
        moving = dev > tol
        if moving.any():
            candidates.append((df["P_bars"] / p_init)[moving].max())
    if not candidates:
        return 1.0
    return min(1.0, max(candidates) * X_PAD)


n_rows, n_cols = len(Y_ROWS), len(SAMPLES)
top_titles = [SAMPLE_DISPLAY_NAMES.get(s, s) for s in SAMPLES]
subplot_titles = top_titles + [""] * ((n_rows - 1) * n_cols)

# shared_xaxes=False so each species/sample panel can carry its own cropped range.
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    shared_xaxes=False, vertical_spacing=0.05, horizontal_spacing=0.05,
    subplot_titles=subplot_titles,
)

for r, (col_name, y_label) in enumerate(Y_ROWS, start=1):
    for c, sample in enumerate(SAMPLES, start=1):
        for tool in TOOLS:
            df = systems.get(sample, {}).get(tool)
            if df is None or col_name not in df.columns or "P_bars" not in df.columns:
                continue
            p_init = df["P_bars"].iloc[0]
            if p_init == 0:
                continue
            x_norm = df["P_bars"] / p_init
            fig.add_trace(
                go.Scatter(
                    mode="lines",
                    x=x_norm, y=df[col_name],
                    name=tool,
                    line=dict(color=TOOL_COLORS_HEX.get(tool, "#333"), width=2,
                              dash=TOOL_LINE_STYLE.get(tool, "solid"),
                              ),
                    showlegend=(r == 1 and c == 1),
                ),
                row=r, col=c,
            )
        fig.update_yaxes(title_text=y_label if c == 1 else None, row=r, col=c)
        fig.update_xaxes(
            title_text="P / P<sub>i</sub>" if r == 3 else None, row=r, col=c,
            range=[0, panel_x_max(sample, col_name)],
        )

legend_style_dict = dict(
    font=dict(size=PLOTLY_LEGEND_FONTSIZE),
    x=0.99, y=0.02,
    xanchor="right", yanchor="bottom",
    bgcolor="white",
    bordercolor="black",
    borderwidth=1,
)

fig.update_layout(
    height=800, width=1000,
    plot_bgcolor="white",
    margin=dict(t=40, r=30, l=60, b=50),
    font=PLOTLY_FONT,
    legend=legend_style_dict,
)
fig.update_xaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
)
fig.update_yaxes(
    showline=True, linewidth=1, linecolor="black", mirror=True,
    ticks="outside", ticklen=PLOTLY_TICK_LEN, tickcolor="black",
    tickfont=dict(size=PLOTLY_TICK_FONTSIZE),
    rangemode="tozero",
)

if SAVE_FIG:
    fig.write_image("figures/Fig_melt_volatiles.png", scale=4)

fig.show()
